In [ ]:
%%capture

!pip install transformers datasets==3.6.0 evaluate

In [ ]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_WRITE"))

In [ ]:
# Если нужно отключить логирование в W&B

import os

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

# Masked language modeling

Masked language modeling предсказывает замаскированный токен в последовательности, и модель может обрабатывать токены в двух направлениях. Это означает, что модель имеет полный доступ к токенам слева и справа. Моделирование языка с маской отлично подходит для задач, которые требуют хорошего контекстного понимания всей последовательности. BERT является примером masked language model.

## Load ELI5 dataset

Начнем с загрузки меньшего подмножества датасета ELI5 из библиотеки 🤗 Datasets. Это даст возможность поэкспериментировать и убедиться, что все работает, прежде чем тратить больше времени на обучение на полном наборе данных.

In [ ]:
from datasets import load_dataset

eli5 = load_dataset("sentence-transformers/eli5", split="train[:5000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pair/train-00000-of-00001.parquet:   0%|          | 0.00/113M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/325475 [00:00<?, ? examples/s]

Разделим набор данных `train_asks` на обучающий и тестовый наборы с помощью метода [train_test_split](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.train_test_split):

In [ ]:
eli5 = eli5.train_test_split(test_size=0.2)

Посмотрим на отдельный элемент датасета

In [ ]:
eli5["train"][0]

{'question': "When Windows first loads and everything is going extremely slowly for several minutes but task manager doesn't show anything taking up much CPU or RAM, what the hell is it doing?",
 'answer': "Look at your I/O (Disk usage), most of the time, that is what is slowing down your windows startup the most.  That's why so many people buy SSDs, they are much faster than regular hard drives (but obviously more expensive)."}

В данном случае нам интересно поле `answer`. Что круто в задачах языкового моделирования, так это то, что нам не нужны метки (фактически мы обучаем модель в unsupervised режиме), потому что следующее слово *является* меткой.

## Предобработка

Для masked language modeling следующим шагом будет загрузка токенизатора DistilRoBERTa для обработки подполя `answer`:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Напишем функцию токенизации ответа

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["answer"])

Чтобы применить эту функцию предобработки ко всему набору данных, используем метод 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map). Вы можете ускорить функцию `map`, установив `batched=True` для одновременной обработки нескольких элементов набора данных и увеличив количество процессов с помощью `num_proc`. Удалим все ненужные столбцы:

In [ ]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Этот набор данных содержит последовательности токенов, но некоторые из них длиннее максимальной входной длины для модели.

Теперь мы можем использовать вторую функцию предварительной обработки, чтобы
- объединить все последовательности
- разделить объединенные последовательности на более короткие фрагменты, определяемые `block_size`, которые должны быть как короче максимальной входной длины, так и достаточно короткими для оперативной памяти графического процессора.

In [ ]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

Применим функцию `group_texts` ко всему набору данных:

In [ ]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Теперь создадим батч примеров с помощью [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling).

Используем токен конца последовательности в качестве токена дополнения и укажем `mlm_probability`, чтобы случайным образом маскировать токены каждый раз, когда итерируемся по данным:

In [ ]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

## Train

Теперь мы готовы начать обучение модели! Загрузим DistilRoBERTa с [AutoModelForMaskedLM](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForMaskedLM):

In [ ]:
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained("distilroberta-base")

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="my_awesome_eli5_mlm_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.142054
2,2.309355,2.137775
3,2.194478,2.120550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1083, training_loss=2.24362165709942, metrics={'train_runtime': 263.0964, 'train_samples_per_second': 32.897, 'train_steps_per_second': 4.116, 'total_flos': 286960434013440.0, 'train_loss': 2.24362165709942, 'epoch': 3.0})

После завершения обучения используем метод [evaluate()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.evaluate), чтобы оценить вашу модель и получить ее перплексию:

In [ ]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 7.96


In [ ]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...m_model/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...m_model/model.safetensors:  12%|#2        | 40.0MB /  329MB            

CommitInfo(commit_url='https://huggingface.co/masterkristall/my_awesome_eli5_mlm_model/commit/69dfc27351e33d56f3d9d7cb969297c59e0fc416', commit_message='End of training', commit_description='', oid='69dfc27351e33d56f3d9d7cb969297c59e0fc416', pr_url=None, repo_url=RepoUrl('https://huggingface.co/masterkristall/my_awesome_eli5_mlm_model', endpoint='https://huggingface.co', repo_type='model', repo_id='masterkristall/my_awesome_eli5_mlm_model'), pr_revision=None, pr_num=None)

In [ ]:
tokenizer.push_to_hub(repo_id='masterkristall/my_awesome_eli5_mlm_model')

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/masterkristall/my_awesome_eli5_mlm_model/commit/69dfc27351e33d56f3d9d7cb969297c59e0fc416', commit_message='Upload tokenizer', commit_description='', oid='69dfc27351e33d56f3d9d7cb969297c59e0fc416', pr_url=None, repo_url=RepoUrl('https://huggingface.co/masterkristall/my_awesome_eli5_mlm_model', endpoint='https://huggingface.co', repo_type='model', repo_id='masterkristall/my_awesome_eli5_mlm_model'), pr_revision=None, pr_num=None)

## Инференс

In [ ]:
text = "The Milky Way is a <mask> galaxy."

In [ ]:
from transformers import pipeline

mask_filler = pipeline("fill-mask", "masterkristall/my_awesome_eli5_mlm_model")
mask_filler(text, top_k=3)

config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

[{'score': 0.6274665594100952,
  'token': 21300,
  'token_str': ' spiral',
  'sequence': 'The Milky Way is a spiral galaxy.'},
 {'score': 0.04613982513546944,
  'token': 2232,
  'token_str': ' massive',
  'sequence': 'The Milky Way is a massive galaxy.'},
 {'score': 0.034755535423755646,
  'token': 30794,
  'token_str': ' dwarf',
  'sequence': 'The Milky Way is a dwarf galaxy.'}]

Токенизируем текст и получим `input_ids` - тензоры PyTorch. Также нужно будет указать позицию токена `<mask>`:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("masterkristall/my_awesome_eli5_mlm_model")
inputs = tokenizer(text, return_tensors="pt")
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

In [ ]:
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained("masterkristall/my_awesome_eli5_mlm_model")
logits = model(**inputs).logits
mask_token_logits = logits[0, mask_token_index, :]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

In [ ]:
top_3_tokens = torch.topk(mask_token_logits, 3, dim=1).indices[0].tolist()

for token in top_3_tokens:
    print(text.replace(tokenizer.mask_token, tokenizer.decode([token])))

The Milky Way is a  spiral galaxy.
The Milky Way is a  massive galaxy.
The Milky Way is a  dwarf galaxy.
